**Saving Trained Models**

Py libraries to save models & pipelines:
- **joblib**
- **pickle**

| Aspect | joblib | pickle |
|------|--------|--------|
| Speed (large models) | Fast | Slow |
| NumPy array handling | Optimized | Not optimized |
| Memory usage | Efficient | High |
| ML pipelines | Ideal | Works but suboptimal |
| Built-in | No | Yes |
| scikit-learn recommended | Yes | No |

**Diabetes Detection - Simplified Version**

In [95]:
%pip install -q numpy pandas matplotlib scikit-learn joblib

In [96]:
# Import the dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report


In [97]:
# configurations
pd.set_option("display.max_columns", None)
pd.reset_option("display.float_format")

# pd.set_option("display.float_format", lambda x: f"{x:.3f}")

RANDOM_STATE = 42
# update with your system path for diabetes.csv
# NOTE: should use backward slash on windows
CSV_PATH="/content/sample_data/diabetes.csv"
TARGET_COL="Outcome"

In [98]:
# load data
df = pd.read_csv(CSV_PATH)
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [99]:
df.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


**Target:**

- 0 - non-diabetic
- 1 - diabetic

In [100]:
print(df[TARGET_COL].value_counts())
print("-"*50)
print(df[TARGET_COL].value_counts(normalize=True)*100)

Outcome
0    500
1    268
Name: count, dtype: int64
--------------------------------------------------
Outcome
0    65.104167
1    34.895833
Name: proportion, dtype: float64


** Imbalanced Classes **

**NOTE:**

`Glucose, BloodPressure, SkinThickness, Insulin, BMI` - These columns have missing values encoded as 0.

In [101]:
# separate features and target
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

In [102]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

In [103]:
print("Train Size:", len(X_train))
print("Test Size:", len(X_test))

Train Size: 614
Test Size: 154


**NOTE: Using Simple Logistic Regression instead of trying to find out the best model as intention is keep it simple and mainly focus on saving the trained model.**

In [104]:
# pipeline = scaler + model
model_pipeline = Pipeline(
    steps=[
         ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced"))
    ]
)

In [105]:
model_pipeline.fit(X_train, y_train)
#

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(class_weight='balanced'))])

In [106]:
# model evaluation
y_train_pred = model_pipeline.predict(X_train)
y_test_pred = model_pipeline.predict(X_test)

# accuracy
train_acc = accuracy_score(y_train, y_train_pred) * 100
test_acc = accuracy_score(y_test, y_test_pred) * 100

# results
print(f"Train Accuracy: {train_acc:.2f}%")
print(f"Test Accuracy: {test_acc:.2f}%\n")

print("-"*40)

print("Train Classification Report")
print(classification_report(y_train, y_train_pred))

print("-"*40)

print("Test Classification Report")
print(classification_report(y_test, y_test_pred))

Train Accuracy: 76.06%
Test Accuracy: 73.38%

----------------------------------------
Train Classification Report
              precision    recall  f1-score   support

           0       0.84      0.78      0.81       400
           1       0.64      0.73      0.68       214

    accuracy                           0.76       614
   macro avg       0.74      0.75      0.74       614
weighted avg       0.77      0.76      0.76       614

----------------------------------------
Test Classification Report
              precision    recall  f1-score   support

           0       0.82      0.75      0.79       100
           1       0.60      0.70      0.65        54

    accuracy                           0.73       154
   macro avg       0.71      0.73      0.72       154
weighted avg       0.75      0.73      0.74       154



##Saving the Test Model.

**joblib**

In [107]:
from joblib import dump

In [108]:
# create model_dir
# path where model file need to be saved
# NOTE: might have to use backslashes on windows;
joblib_path = "/content/sample_data/diabetes_model_pipeline.joblib"

In [109]:
dump(model_pipeline, joblib_path)

['/content/sample_data/diabetes_model_pipeline.joblib']

**Pickle**

In [110]:
import pickle

In [111]:
pickle_path = "/content/sample_data/diabetes_model_pipeline.pkl"

In [112]:
with open (pickle_path,"wb") as f:
  pickle.dump(model_pipeline, f)
#